# Deep Ensembles under 30% missingness (capped RUL, matches `dropout_missing_FIXED_v3.ipynb` convention)

Same masking setup as the MC-Dropout missing-data notebook -- 34 input features (17 raw + 17 observed/missing indicators) -- but with 5 independently-seeded BiLSTM models instead of one model queried with dropout on.

In [1]:
import pandas as pd
import numpy as np

FEATURES = [
    "Setting_1",
    "Setting_2",
    "Sensor_2",
    "Sensor_3",
    "Sensor_4",
    "Sensor_6",
    "Sensor_7",
    "Sensor_8",
    "Sensor_9",
    "Sensor_11",
    "Sensor_12",
    "Sensor_13",
    "Sensor_14",
    "Sensor_15",
    "Sensor_17",
    "Sensor_20",
    "Sensor_21"
]

TARGET = "RUL"


def create_masked_dataset(df, features):

    data = df.copy()
    mask_features = []

    for feature in features:
        mask_name = feature + "_mask"

        # 1 = observed, 0 = missing
        data[mask_name] = data[feature].notna().astype("float32")
        mask_features.append(mask_name)

    # Placeholder only -- the mask column tells the model whether 0 is a
    # real reading or a stand-in for missing data.
    data[features] = data[features].fillna(0.0).astype("float32")

    return data, mask_features


In [2]:
train_missing = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone_updated\Dataset\Missing_Train\processed_train_missing_30.csv")
test_missing = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone_updated\Dataset\Missing_Test\processed_test_missing_30.csv")

# train_missing["RUL"] is already capped at 125 (inherited from the capped
# processed_train.csv that data_missing_FIXED.ipynb was run against).

train_masked, MASK_FEATURES = create_masked_dataset(train_missing, FEATURES)
test_masked, _ = create_masked_dataset(test_missing, FEATURES)

MODEL_FEATURES = FEATURES + MASK_FEATURES
print("Model features (raw + mask):", len(MODEL_FEATURES))

np.random.seed(42)

engine_ids = train_masked["Engine_ID"].unique()
np.random.shuffle(engine_ids)

train_ids = engine_ids[:70]
val_ids = engine_ids[70:85]
test_ids = engine_ids[85:]

train_df = train_masked[train_masked["Engine_ID"].isin(train_ids)].copy()
val_df = train_masked[train_masked["Engine_ID"].isin(val_ids)].copy()
test_df = train_masked[train_masked["Engine_ID"].isin(test_ids)].copy()

print("Training engines:", train_df["Engine_ID"].nunique())
print("Validation engines:", val_df["Engine_ID"].nunique())
print("Testing engines:", test_df["Engine_ID"].nunique())
print("\nCapped RUL range (train):", train_masked["RUL"].min(), "-", train_masked["RUL"].max())


Model features (raw + mask): 34
Training engines: 70
Validation engines: 15
Testing engines: 15

Capped RUL range (train): 0 - 125


In [3]:
WINDOW = 30

def create_sequences(data, features, target, window=30):
    X = []
    y = []

    for engine_id, engine_data in data.groupby("Engine_ID"):
        engine_data = engine_data.sort_values("Cycle")

        feature_values = engine_data[features].values.astype("float32")
        target_values = engine_data[target].values.astype("float32")

        for i in range(window, len(engine_data)):
            X.append(feature_values[i-window:i])
            y.append(target_values[i])

    return np.array(X), np.array(y)


In [4]:
X_train, y_train = create_sequences(train_df, MODEL_FEATURES, TARGET, WINDOW)
X_val, y_val = create_sequences(val_df, MODEL_FEATURES, TARGET, WINDOW)
X_test, y_test = create_sequences(test_df, MODEL_FEATURES, TARGET, WINDOW)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)


(12216, 30, 34)
(2720, 30, 34)
(2695, 30, 34)


## Train the ensemble (input width = 34)

In [5]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Bidirectional,
    Dense,
    Dropout
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


def build_bilstm(seed):
    tf.random.set_seed(seed)

    model = Sequential([

        Input(shape=(WINDOW, len(MODEL_FEATURES))),

        Bidirectional(
            LSTM(
                64,
                return_sequences=True
            )
        ),

        Dropout(0.2),

        Bidirectional(
            LSTM(32)
        ),

        Dropout(0.2),

        Dense(
            32,
            activation="relu"
        ),

        Dense(1)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model


C:\Users\Shivam\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.2) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [6]:
N_MODELS = 5
SEEDS = [42, 43, 44, 45, 46]

ensemble_models = []

for seed in SEEDS:

    tf.keras.backend.clear_session()

    print(f"\n--- Training ensemble member, seed={seed} ---")

    model = build_bilstm(seed)

    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )

    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6
    )

    model.fit(
        X_train,
        y_train,

        validation_data=(
            X_val,
            y_val
        ),

        epochs=100,
        batch_size=64,

        shuffle=False,

        callbacks=[
            early_stopping,
            reduce_lr
        ],

        verbose=1
    )

    ensemble_models.append(model)

print(f"\nTrained {len(ensemble_models)} ensemble members.")




--- Training ensemble member, seed=42 ---
Epoch 1/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 10s 27ms/step - loss: 3709.8406 - mae: 51.2468 - val_loss: 1757.2334 - val_mae: 37.4834 - learning_rate: 0.0010
Epoch 2/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - loss: 1775.4644 - mae: 37.3570 - val_loss: 1748.8677 - val_mae: 37.1093 - learning_rate: 0.0010
Epoch 3/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - loss: 1777.7821 - mae: 37.3149 - val_loss: 1748.8104 - val_mae: 37.1082 - learning_rate: 0.0010
Epoch 4/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - loss: 1777.4037 - mae: 37.3045 - val_loss: 1748.7915 - val_mae: 37.0978 - learning_rate: 0.0010
Epoch 5/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - loss: 1778.4049 - mae: 37.3177 - val_loss: 1747.1915 - val_mae: 37.1072 - learning_rate: 0.0010
Epoch 6/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - loss: 1534.1306 - mae: 34.2938 - val_loss: 873.0284 - val_mae: 25.7744 - learning_rate: 0.0010
Epoch 7/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 

## Ensemble prediction

In [7]:
def ensemble_predict(models, X):

    predictions = np.array([
        model.predict(X, verbose=0).ravel()
        for model in models
    ])

    mean_pred = predictions.mean(axis=0)
    std_pred = predictions.std(axis=0)

    return mean_pred, std_pred


In [8]:
ens_mean, ens_std = ensemble_predict(
    ensemble_models,
    X_test
)

print("Predictions:", ens_mean.shape)
print("Uncertainty:", ens_std.shape)

print("\nMean uncertainty:", ens_std.mean())
print("Min uncertainty :", ens_std.min())
print("Max uncertainty :", ens_std.max())


Predictions: (2695,)
Uncertainty: (2695,)

Mean uncertainty: 5.9347324
Min uncertainty : 0.44990426
Max uncertainty : 25.05113


In [9]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Held-out split:")
print("MAE :", mean_absolute_error(y_test, ens_mean))
print("RMSE:", np.sqrt(mean_squared_error(y_test, ens_mean)))
print("R\u00b2  :", r2_score(y_test, ens_mean))


Held-out split:
MAE : 13.792115211486816
RMSE: 17.85797631634767
R²  : 0.8174680471420288


## Calibrate the accept/abstain threshold on `val_df`

In [10]:
ens_cal_mean, ens_cal_std = ensemble_predict(
    ensemble_models,
    X_val
)

ACCEPT_RATE = 0.80

ens_threshold = np.quantile(
    ens_cal_std,
    ACCEPT_RATE
)

print("Ensemble uncertainty threshold:", ens_threshold)


Ensemble uncertainty threshold: 8.937666


## Official test set (capped, 30% missing -- literature-comparable convention)

In [11]:
RUL_CAP = 125

output_df = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone_updated\Dataset\RUL_FD001.csv")
output_df["Engine_ID"] = np.arange(1, len(output_df) + 1)
output_df["RUL"] = output_df["RUL"].clip(upper=RUL_CAP)

def create_test_sequences(test_df, features, window=30):
    X, engine_ids = [], []
    for engine_id, engine_data in test_df.groupby("Engine_ID"):
        engine_data = engine_data.sort_values("Cycle")
        values = engine_data[features].values.astype("float32")
        if len(values) >= window:
            X.append(values[-window:])
            engine_ids.append(engine_id)
    return np.array(X), np.array(engine_ids)

X_test_final, test_engine_ids = create_test_sequences(test_masked, MODEL_FEATURES, WINDOW)


In [12]:
ens_test_mean, ens_test_std = ensemble_predict(
    ensemble_models,
    X_test_final
)

pred_df = pd.DataFrame({
    "Engine_ID": test_engine_ids,
    "Predicted_RUL": ens_test_mean,
    "Uncertainty": ens_test_std
})

comparison = pred_df.merge(output_df, on="Engine_ID")

actual = comparison["RUL"].values
ens_test_mean = comparison["Predicted_RUL"].values
ens_test_std = comparison["Uncertainty"].values


In [13]:
def nasa_score(actual, predicted):
    """NASA / PHM08 asymmetric RUL scoring function (Saxena et al., 2008).
    Penalizes late predictions (predicted RUL > actual RUL) more heavily
    than early ones, since a late prediction means maintenance would be
    scheduled after the engine could already have failed.
    """
    d = np.asarray(predicted) - np.asarray(actual)
    s = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    return np.sum(s)


In [14]:
print("100% coverage:")
print("MAE  :", mean_absolute_error(actual, ens_test_mean))
print("RMSE :", np.sqrt(mean_squared_error(actual, ens_test_mean)))
print("R\u00b2   :", r2_score(actual, ens_test_mean))
print("Score:", nasa_score(actual, ens_test_mean))


100% coverage:
MAE  : 11.013370513916016
RMSE : 14.261760476520717
R²   : 0.8733408451080322
Score: 362.425632380351


## Accept / abstain sweep

In [15]:
accept = ens_test_std <= ens_threshold

print("Accepted :", accept.sum())
print("Abstained:", (~accept).sum())
print("Coverage :", accept.mean())


Accepted : 78
Abstained: 22
Coverage : 0.78


In [16]:
y_true_accept = actual[accept]
y_pred_accept = ens_test_mean[accept]

mae = mean_absolute_error(y_true_accept, y_pred_accept)
rmse = np.sqrt(mean_squared_error(y_true_accept, y_pred_accept))
r2 = r2_score(y_true_accept, y_pred_accept)
score = nasa_score(y_true_accept, y_pred_accept)

print("Accepted predictions:")
print("MAE  :", mae)
print("RMSE :", rmse)
print("R\u00b2   :", r2)
print("Score:", score)


Accepted predictions:
MAE  : 9.807374000549316
RMSE : 13.070506877103659
R²   : 0.9070001244544983
Score: 241.41832680298342


In [17]:
for rate in [0.9, 0.8, 0.7]:

    threshold = np.quantile(
        ens_cal_std,
        rate
    )

    accept = ens_test_std <= threshold

    rmse = np.sqrt(
        mean_squared_error(
            actual[accept],
            ens_test_mean[accept]
        )
    )

    score = nasa_score(
        actual[accept],
        ens_test_mean[accept]
    )

    print(
        f"Acceptance={rate:.0%} | "
        f"Accepted={accept.sum()} | "
        f"RMSE={rmse:.3f} | "
        f"Score={score:.1f}"
    )


Acceptance=90% | Accepted=89 | RMSE=13.267 | Score=280.7
Acceptance=80% | Accepted=78 | RMSE=13.071 | Score=241.4
Acceptance=70% | Accepted=70 | RMSE=13.108 | Score=225.0


In [18]:
import os

METHOD = "deep_ensembles"
SCENARIO = "missing_capped"

RESULTS_DIR = r"C:\Users\Shivam\Desktop\Capstone_updated\results"
os.makedirs(RESULTS_DIR, exist_ok=True)

predicted = ens_test_mean
uncertainty = ens_test_std

results_df = pd.DataFrame({
    "Engine_ID": test_engine_ids,
    "Actual_RUL": actual,
    "Predicted_RUL": predicted,
    "Uncertainty": uncertainty
})

out_path = os.path.join(
    RESULTS_DIR,
    f"{METHOD}__{SCENARIO}.csv"
)

results_df.to_csv(out_path, index=False)

print(f"Saved {len(results_df)} rows -> {out_path}")

Saved 100 rows -> C:\Users\Shivam\Desktop\Capstone_updated\results\deep_ensembles__missing_capped.csv
